# Load needed modules

In [1]:
# modules:
import pandas as pd
import os
from openai import OpenAI


# get external variables or functions:
import src.API_key as key


from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from langchain.callbacks import get_openai_callback


import json


# Load data to translate
> this should be the drawn concept data set whereby 1 row is 1 drawn concept

In [4]:
print(os.getcwd())
# Get the current working directory
os.chdir("outputs/associations") 
directory = os.getcwd()

/home/fenn/Desktop/Publications/Article_HumanOversightAss/Article_humanOversightAssociations/Analyses


In [5]:
# print the current working directory
print(directory)

/home/fenn/Desktop/Publications/Article_HumanOversightAss/Article_humanOversightAssociations/Analyses/outputs/associations


In [6]:
# List files in the current working directory
files = os.listdir('.')
# Display the list of files
print(files)

['placeholder.txt', 'ass_study.xlsx', 'ass_study.csv', 'ass_study.rds']


In [7]:
# Load the .xlsx file into a DataFrame
df = pd.read_excel(directory + "/ass_study.xlsx")

# prepare the data frames 
# for the text only the unique values are kept
print("Number of rows/ colums:")
print(df.shape)

Number of rows/ colums:
(18960, 11)


In [9]:
df

,participant_id,study_condition,gender,age,cue,valence,response,response_position,response_level,timestamp,time_diff_sec
0,6914df322c0ecd3bdadf7251,Diagnostic,male,33.0,Diagnostic,6.0,Richtig,1,1,2025-11-24 08:16:27.162,0.000
1,6914df322c0ecd3bdadf7251,Diagnostic,male,33.0,Diagnostic,7.0,Fortschritt,2,1,2025-11-24 08:16:35.087,7.925
2,6914df322c0ecd3bdadf7251,Diagnostic,male,33.0,Diagnostic,5.0,Zukunft,3,1,2025-11-24 08:16:38.866,11.704
3,6914df322c0ecd3bdadf7251,Diagnostic,male,33.0,Diagnostic,6.0,Zusammenspiel,4,1,2025-11-24 08:16:46.704,19.542
4,6914df322c0ecd3bdadf7251,Diagnostic,male,33.0,Diagnostic,7.0,Verbesserung,5,1,2025-11-24 08:16:54.584,27.422
...,...,...,...,...,...,...,...,...,...,...,...
18955,69273d7a43eca09e260bac35,CourtDecision,female,22.0,Fragwürdig,NaN,wieso,1,2,2026-02-11 17:13:58.343,443.097
18956,69273d7a43eca09e260bac35,CourtDecision,female,22.0,Fragwürdig,NaN,künstlich,2,2,2026-02-11 17:14:07.421,452.175
18957,69273d7a43eca09e260bac35,CourtDecision,female,22.0,Fragwürdig,NaN,unsicher,3,2,2026-02-11 17:14:15.217,459.971
18958,69273d7a43eca09e260bac35,CourtDecision,female,22.0,Fragwürdig,NaN,schlecht,4,2,2026-02-11 17:14:22.131,466.885


# test dictionary

In [10]:
# Exemplary dictionary
dictionary = {
    'Hallo': 'Hello',
    'Guten Tag': 'Good day',
    'Wie sagst es dir?': 'How are you?',
    'Ich bind eine Roboter.': 'I am a robot.',
    'Ich bin ein sozialer Flofods.': 'I am a social Flofods.'
}

dictionary.update({'AAA': 'BBB',  'Hallo': 'Hello',  'aaa': '111'})

# Convert the dictionary into a DataFrame
df_test = pd.DataFrame(list(dictionary.items()), columns=['German', 'English'])

# Display the DataFrame
print(df_test)

print(dictionary.keys())

                          German                 English
0                          Hallo                   Hello
1                      Guten Tag                Good day
2              Wie sagst es dir?            How are you?
3         Ich bind eine Roboter.           I am a robot.
4  Ich bin ein sozialer Flofods.  I am a social Flofods.
5                            AAA                     BBB
6                            aaa                     111
dict_keys(['Hallo', 'Guten Tag', 'Wie sagst es dir?', 'Ich bind eine Roboter.', 'Ich bin ein sozialer Flofods.', 'AAA', 'aaa'])


# Set up ChatGPT

## Translate written text new

In [11]:
json_schema = {
    "title": "Translation",
    "description": "Dictionary that contains the words and their respective translations.",
    "type": "object",
    "properties": {
        "dictionary": {
            "type": "string",
            "description": 'Dictionary that contains the words as keys and their respective translations as values. Provide the dictionary in the following format: {"word1": "translation1", "word2": "translation2", ...}.',
        },
    },
    "required": ["dictionary"],
}

In [12]:
system_template = """<Context>You are a helpful assistant that translates {language} words to English.</Context>

<Data Structure>The lists "translate" is an array containing {language} words or sentences.</Data Structure>

<Task>Translate the provided array "translate" to English, whereby you return a dictionary with the {language} words as keys and their English translations as values. 
The dictionary should be structured according to the provided JSON schema.
Be careful with spelling errors and do not use capital letters for adjectives.
Repeat this process in case no translations are returned or the dictionary do not follow the JSON schema.</Task>
"""

user_template = """Array "translate": 
{translate}
"""

# rescue robots and socially assistive robots
prompt_template = ChatPromptTemplate.from_messages(
    [("system", system_template), ("user", user_template)]
)


result = prompt_template.invoke({"language": "German", "translate":["Hallo", "Guten Tag"]})
print(result)

print("result:", result)
print("result.to_messages():", result.to_messages())

messages=[SystemMessage(content='<Context>You are a helpful assistant that translates German words to English.</Context>\n\n<Data Structure>The lists "translate" is an array containing German words or sentences.</Data Structure>\n\n<Task>Translate the provided array "translate" to English, whereby you return a dictionary with the German words as keys and their English translations as values. \nThe dictionary should be structured according to the provided JSON schema.\nBe careful with spelling errors and do not use capital letters for adjectives.\nRepeat this process in case no translations are returned or the dictionary do not follow the JSON schema.</Task>\n', additional_kwargs={}, response_metadata={}), HumanMessage(content='Array "translate": \n[\'Hallo\', \'Guten Tag\']\n', additional_kwargs={}, response_metadata={})]
result: messages=[SystemMessage(content='<Context>You are a helpful assistant that translates German words to English.</Context>\n\n<Data Structure>The lists "transla

In [14]:
def basic_API_call(
    prompt,
    language,
    translate,
    openai_api_key,
    json_schema,
    model_name="gpt-4o",
    max_tokens=1000,
    verbose=False
):

    # prompt = PromptTemplate(template=template)
    seed = 123

    model = ChatOpenAI(model=model_name, openai_api_key=openai_api_key, max_tokens=max_tokens, model_kwargs={"seed": seed}, temperature=0.0)
       
    structured_llm = model.with_structured_output(json_schema, include_raw=True)
    chain = prompt | structured_llm

    with get_openai_callback() as cb:
        response = chain.invoke(
            {"language": language, "translate": translate}
        )
        print(cb)
    
    if verbose:
        print(f"Total Tokens: {cb.total_tokens}")
        print(f"Prompt Tokens: {cb.prompt_tokens}")
        print(f"Completion Tokens: {cb.completion_tokens}")
        print(f"Total Cost (USD): ${cb.total_cost}")
        
    return response

In [15]:
result = basic_API_call(prompt=prompt_template,
    language="German",
    translate=["Hallo", "Guten Tag", "Wie geht es dir?"],
    openai_api_key=key.openai_api_key,
    json_schema=json_schema,
    model_name="gpt-4o",
    max_tokens=1000
)

/home/fenn/Documents/env_python/lib/python3.12/site-packages/IPython/core/interactiveshell.py:3577: UserWarning: Parameters {'seed'} should be specified explicitly. Instead they were passed in as part of `model_kwargs` parameter.
  exec(code_obj, self.user_global_ns, self.user_ns)


Tokens Used: 262
	Prompt Tokens: 229
	Completion Tokens: 33
Successful Requests: 1
Total Cost (USD): $0.0009025


In [16]:
# Extract the 'parsed' section from the JSON data
parsed_section = result.get('parsed', {})
dictionary = parsed_section.get('dictionary')
    
# Convert the string back to a Python dictionary
print(type(dictionary)) 
dictionary = eval(dictionary)
print(len(dictionary.keys()))
print(dictionary)

<class 'str'>
3
{'Hallo': 'Hello', 'Guten Tag': 'Good day', 'Wie geht es dir?': 'How are you?'}


### Apply code

In [17]:
print(os.getcwd())
os.chdir("../associations_translated")
print(os.getcwd())

/home/fenn/Desktop/Publications/Article_HumanOversightAss/Article_humanOversightAssociations/Analyses/outputs/associations
/home/fenn/Desktop/Publications/Article_HumanOversightAss/Article_humanOversightAssociations/Analyses/outputs/associations_translated


In [20]:
print("Number of rows/ colums:")
print(df.shape)
print(df["response"].unique().shape)

Number of rows/ colums:
(18960, 11)
(9856,)


In [22]:
def translate_response_column(df, chunk_size=8, language="German", target_column="response"):
    """
    Translate the response column from German to English and add as response_translated.
    
    Parameters:
    - df: DataFrame containing the data
    - chunk_size: Number of texts to translate in each API call
    - language: Source language (default: German)
    - target_column: Column to translate (default: response)
    
    Returns:
    - DataFrame with added response_translated column
    """
    
    # Make a copy of the dataframe to avoid modifying the original
    df_copy = df.copy()
    
    # Check if target column exists
    if target_column not in df_copy.columns:
        print(f"Column '{target_column}' not found in DataFrame.")
        return df_copy
    
    # Get non-missing values from the response column
    non_missing_responses = df_copy[target_column].dropna()
    
    if non_missing_responses.empty:
        print(f"No non-missing values found in column '{target_column}'.")
        df_copy[f"{target_column}_translated"] = None
        return df_copy
    
    # Get unique texts to minimize API calls
    unique_texts = non_missing_responses.astype(str).unique().tolist()
    print(f"Translating {len(unique_texts)} unique texts from {language} to English...")
    
    # Dictionary to store translations
    translation_dict = {}
    counter = 0
    
    # Process texts in chunks
    for i in range(0, len(unique_texts), chunk_size):
        counter += 1
        chunk = unique_texts[i:i + chunk_size]
        print(f"Translating chunk {counter} ({len(chunk)} texts)...")
        
        try:
            result = basic_API_call(
                prompt=prompt_template,
                language=language,
                translate=chunk,
                openai_api_key=key.openai_api_key,
                json_schema=json_schema,
                model_name="gpt-4o",
                max_tokens=1000,
                verbose=False
            )
            
            parsed_section = result.get("parsed", {})
            dictionary_str = parsed_section.get("dictionary")
            
            if dictionary_str:
                # Convert string dictionary to actual dictionary
                chunk_translations = eval(dictionary_str)
                translation_dict.update(chunk_translations)
                print(f"Successfully translated chunk {counter}")
            else:
                print(f"No translations returned for chunk {counter}")
                
        except Exception as e:
            print(f"Error translating chunk {counter}: {e}")
            continue
    
    # Map translations back to the dataframe
    df_copy[f"{target_column}_translated"] = df_copy[target_column].map(translation_dict)
    
    # Report results
    successful_translations = df_copy[f"{target_column}_translated"].notna().sum()
    total_non_missing = non_missing_responses.count()
    print(f"Translation complete: {successful_translations}/{total_non_missing} responses translated")
    
    return df_copy

# Apply the translation function to the dataframe
print("Starting translation of response column...")
df_translated = translate_response_column(df, chunk_size=60, language="German", target_column="response")

# Display results
print("\nTranslation results:")
print(f"Original dataframe shape: {df.shape}")
print(f"Translated dataframe shape: {df_translated.shape}")
print(f"New column added: response_translated")

# Show sample of translations
if 'response_translated' in df_translated.columns:
    sample_translations = df_translated[['response', 'response_translated']].dropna().head(10)
    print("\nSample translations:")
    for idx, row in sample_translations.iterrows():
        print(f"German: {row['response']}")
        print(f"English: {row['response_translated']}")
        print("---")

Starting translation of response column...
Translating 9856 unique texts from German to English...
Translating chunk 1 (60 texts)...


/tmp/ipykernel_200739/802118741.py:84: UserWarning: Parameters {'seed'} should be specified explicitly. Instead they were passed in as part of `model_kwargs` parameter.
  df_translated = translate_response_column(df, chunk_size=60, language="German", target_column="response")


Tokens Used: 1022
	Prompt Tokens: 507
	Completion Tokens: 515
Successful Requests: 1
Total Cost (USD): $0.0064175
Successfully translated chunk 1
Translating chunk 2 (60 texts)...


/tmp/ipykernel_200739/802118741.py:84: UserWarning: Parameters {'seed'} should be specified explicitly. Instead they were passed in as part of `model_kwargs` parameter.
  df_translated = translate_response_column(df, chunk_size=60, language="German", target_column="response")


Tokens Used: 1194
	Prompt Tokens: 568
	Completion Tokens: 626
Successful Requests: 1
Total Cost (USD): $0.007679999999999999
Successfully translated chunk 2
Translating chunk 3 (60 texts)...


...

Tokens Used: 1359
	Prompt Tokens: 621
	Completion Tokens: 738
Successful Requests: 1
Total Cost (USD): $0.0089325
Successfully translated chunk 163
Translating chunk 164 (60 texts)...


/tmp/ipykernel_200739/802118741.py:84: UserWarning: Parameters {'seed'} should be specified explicitly. Instead they were passed in as part of `model_kwargs` parameter.
  df_translated = translate_response_column(df, chunk_size=60, language="German", target_column="response")


Tokens Used: 1083
	Prompt Tokens: 524
	Completion Tokens: 559
Successful Requests: 1
Total Cost (USD): $0.006900000000000001
Successfully translated chunk 164
Translating chunk 165 (16 texts)...


/tmp/ipykernel_200739/802118741.py:84: UserWarning: Parameters {'seed'} should be specified explicitly. Instead they were passed in as part of `model_kwargs` parameter.
  df_translated = translate_response_column(df, chunk_size=60, language="German", target_column="response")


Tokens Used: 450
	Prompt Tokens: 296
	Completion Tokens: 154
Successful Requests: 1
Total Cost (USD): $0.00228
Successfully translated chunk 165
Translation complete: 18495/18960 responses translated

Translation results:
Original dataframe shape: (18960, 11)
Translated dataframe shape: (18960, 12)
New column added: response_translated

Sample translations:
German: Richtig
English: correct
---
German: Fortschritt
English: progress
---
German: Zukunft
English: future
---
German: Zusammenspiel
English: interaction
---
German: Verbesserung
English: improvement
---
German: Moralisch
English: moral
---
German: Bessere Überlebenschancen
English: better survival chances
---
German: Früherkennung
English: early detection
---
German: Gleichheit
English: equality
---
German: Niveau Steigerung
English: level increase
---


In [33]:
backup_df_translated = df_translated
df_translated

,participant_id,study_condition,gender,age,cue,valence,response,response_position,response_level,timestamp,time_diff_sec,response_translated
0,6914df322c0ecd3bdadf7251,Diagnostic,male,33.0,Diagnostic,6.0,Richtig,1,1,2025-11-24 08:16:27.162,0.000,correct
1,6914df322c0ecd3bdadf7251,Diagnostic,male,33.0,Diagnostic,7.0,Fortschritt,2,1,2025-11-24 08:16:35.087,7.925,progress
2,6914df322c0ecd3bdadf7251,Diagnostic,male,33.0,Diagnostic,5.0,Zukunft,3,1,2025-11-24 08:16:38.866,11.704,future
3,6914df322c0ecd3bdadf7251,Diagnostic,male,33.0,Diagnostic,6.0,Zusammenspiel,4,1,2025-11-24 08:16:46.704,19.542,interaction
4,6914df322c0ecd3bdadf7251,Diagnostic,male,33.0,Diagnostic,7.0,Verbesserung,5,1,2025-11-24 08:16:54.584,27.422,improvement
...,...,...,...,...,...,...,...,...,...,...,...,...
18955,69273d7a43eca09e260bac35,CourtDecision,female,22.0,Fragwürdig,NaN,wieso,1,2,2026-02-11 17:13:58.343,443.097,NaN
18956,69273d7a43eca09e260bac35,CourtDecision,female,22.0,Fragwürdig,NaN,künstlich,2,2,2026-02-11 17:14:07.421,452.175,artificial
18957,69273d7a43eca09e260bac35,CourtDecision,female,22.0,Fragwürdig,NaN,unsicher,3,2,2026-02-11 17:14:15.217,459.971,NaN
18958,69273d7a43eca09e260bac35,CourtDecision,female,22.0,Fragwürdig,NaN,schlecht,4,2,2026-02-11 17:14:22.131,466.885,bad


In [40]:
cols = ["response", "response_translated"]
missing_counts = df_translated[cols].isna().sum()
missing_pct = df_translated[cols].isna().mean() * 100

print("Missing counts and percentages (of rows):")
for c in cols:
    print(f"{c}: {missing_counts[c]} / {len(df_translated)} ({missing_pct[c]:.2f}%)")

# Show up to 10 examples of rows with missing values for each column in `cols`
for c in cols:
    miss = df_translated[df_translated[c].isna()]
    n = len(miss)
    print(f"\nColumn '{c}' — missing {n}/{len(df_translated)} rows ({missing_pct[c]:.2f}%)")
    if n == 0:
        print("  (none)")
    else:
        # show context columns and up to 10 examples
        print(miss.loc[:, ['participant_id', 'cue', 'response', 'response_translated']].head(10).to_string(index=True))

Missing counts and percentages (of rows):
response: 0 / 18960 (0.00%)
response_translated: 190 / 18960 (1.00%)

Column 'response' — missing 0/18960 rows (0.00%)
  (none)

Column 'response_translated' — missing 190/18960 rows (1.00%)
                 participant_id                    cue                response response_translated
11586  6931b0a54a97529bbae235d9               Schlecht  keine Kritikfähigkeit                  NaN
11590  6931b0a54a97529bbae235d9                Ungenau        KI hallizuniert                  NaN
11595  6931b0a54a97529bbae235d9  Zu positive bewertung            Verschönung                  NaN
11596  6931b0a54a97529bbae235d9  Zu positive bewertung            Ehrlichkeit                  NaN
11598  6931b0a54a97529bbae235d9  Zu positive bewertung       Kritikunfähikeit                  NaN
11611  693087d06909b0ac1158bc5d                Grading             Vorsichtig                  NaN
11612  693087d06909b0ac1158bc5d                Grading                 Reg

In [32]:
# Save df_translated to current working directory as XLSX and CSV
filename_base = "ass_study_translated"
xlsx_path = os.path.join(os.getcwd(), f"{filename_base}.xlsx")
csv_path = os.path.join(os.getcwd(), f"{filename_base}.csv")

try:
    df_translated.to_excel(xlsx_path, index=False)
    print(f"Saved Excel: {xlsx_path} ({df_translated.shape[0]} rows, {df_translated.shape[1]} cols)")
except Exception as e:
    print(f"Error saving Excel: {e}")

try:
    df_translated.to_csv(csv_path, index=False)
    print(f"Saved CSV:   {csv_path} ({df_translated.shape[0]} rows, {df_translated.shape[1]} cols)")
except Exception as e:
    print(f"Error saving CSV: {e}")

Saved Excel: /home/fenn/Desktop/Publications/Article_HumanOversightAss/Article_humanOversightAssociations/Analyses/outputs/associations_translated/ass_study_translated.xlsx (18960 rows, 12 cols)
Saved CSV:   /home/fenn/Desktop/Publications/Article_HumanOversightAss/Article_humanOversightAssociations/Analyses/outputs/associations_translated/ass_study_translated.csv (18960 rows, 12 cols)
